# Kenya Weather AgroClimate Data
## 1. Extract the data
Run an extraction pipeline to fetch data from the NASApower website to our local directory

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# from etl.extract import run_data_extraction

# run_data_extraction()

Here we check how many files we extracted and are present in our directory

In [ ]:
from etl.extract import get_county_files

for county in ["Transzoia", "Uasin Gishu", "Nakuru", "Bungoma"]:
    county = county.lower().replace(" ", "_") 
    files = get_county_files(county)
    print(f"{len(files)} files found for {county}:")

4 files found for transzoia:
4 files found for uasin_gishu:
4 files found for nakuru:
4 files found for bungoma:


# Exploring data
After loading the data to our folder, we want to have a view of it so we know what actions to perfom on it.

In [4]:
import pandas as pd
from pathlib import Path
import json

Getting the data from the files inside the data/raw file

In [5]:
# Loop through each file per county confirming the data structure and printing the keys
raw_dir = Path("data/raw")

for file in raw_dir.glob("*.json"):
    print(f"Reading file: {file}")
    with open(file, "r") as f:
        data = json.load(f)

    # Convert the JSON data to a DataFrame
print(type(data))
print(list(data.keys()))

Reading file: data\raw\bungoma_2023.json
Reading file: data\raw\bungoma_2024.json
Reading file: data\raw\bungoma_2025.json
Reading file: data\raw\bungoma_2026.json
Reading file: data\raw\nakuru_2023.json
Reading file: data\raw\nakuru_2024.json
Reading file: data\raw\nakuru_2025.json
Reading file: data\raw\nakuru_2026.json
Reading file: data\raw\transzoia_2023.json
Reading file: data\raw\transzoia_2024.json
Reading file: data\raw\transzoia_2025.json
Reading file: data\raw\transzoia_2026.json
Reading file: data\raw\trans_nzoia_2023.json
Reading file: data\raw\trans_nzoia_2024.json
Reading file: data\raw\trans_nzoia_2025.json
Reading file: data\raw\trans_nzoia_2026.json
Reading file: data\raw\uasin_gishu_2023.json
Reading file: data\raw\uasin_gishu_2024.json
Reading file: data\raw\uasin_gishu_2025.json
Reading file: data\raw\uasin_gishu_2026.json
<class 'dict'>
['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'WS2M', 'ALLSKY_SFC_SW_DWN', 'RH2M', 'GWETTOP']


In [6]:
print(data)

{'T2M': {'20260101': 20.01, '20260102': 20.35, '20260103': 20.47, '20260104': 20.15, '20260105': 19.93, '20260106': 19.33, '20260107': 19.96, '20260108': 19.77, '20260109': 19.15, '20260110': 20.01, '20260111': 19.85, '20260112': 20.47, '20260113': 19.71, '20260114': 18.28, '20260115': 19.17, '20260116': 20.0, '20260117': 20.33, '20260118': 20.42, '20260119': 20.8, '20260120': 20.84, '20260121': 20.56, '20260122': 20.17, '20260123': 20.3, '20260124': 20.37, '20260125': 20.99, '20260126': 21.5, '20260127': 21.09, '20260128': 20.69, '20260129': 20.77, '20260130': 20.83, '20260131': 21.01, '20260201': 21.36, '20260202': 20.93, '20260203': 20.89, '20260204': 21.19, '20260205': 21.71, '20260206': 20.99, '20260207': 20.77, '20260208': 20.89, '20260209': 21.06, '20260210': 20.78, '20260211': 21.22, '20260212': 21.04, '20260213': 21.34, '20260214': 21.75, '20260215': 20.84, '20260216': 19.44, '20260217': 19.0, '20260218': 20.9, '20260219': 19.1, '20260220': 19.86, '20260221': 19.77, '20260222'

Checking whether all parameters have the same shape and data range

In [7]:
for param, values in data.items():
    dates = list(values.keys())
    print(f"Parameter: {param}, Number of entries: {len(dates)}")

Parameter: T2M, Number of entries: 212
Parameter: T2M_MAX, Number of entries: 212
Parameter: T2M_MIN, Number of entries: 212
Parameter: PRECTOTCORR, Number of entries: 212
Parameter: WS2M, Number of entries: 212
Parameter: ALLSKY_SFC_SW_DWN, Number of entries: 212
Parameter: RH2M, Number of entries: 212
Parameter: GWETTOP, Number of entries: 212


Scanning for null values

In [8]:
missing_data = {param: {date: value for date, value in values.items() if value is None} for param, values in data.items()}
for param, missing in missing_data.items():
    print(f"Parameter: {param}, Missing entries: {len(missing)}")

Parameter: T2M, Missing entries: 0
Parameter: T2M_MAX, Missing entries: 0
Parameter: T2M_MIN, Missing entries: 0
Parameter: PRECTOTCORR, Missing entries: 0
Parameter: WS2M, Missing entries: 0
Parameter: ALLSKY_SFC_SW_DWN, Missing entries: 0
Parameter: RH2M, Missing entries: 0
Parameter: GWETTOP, Missing entries: 0


Once loaded lets explore the shape, nulls and number of rows and columns

In [9]:
# Check the shape and structure of the data for each parameter and each town
for param, values in data.items():
    print(f"Parameter: {param}, Number of towns: {len(values)}")

    sample = list(values.items())[:2]
    print(f" sample: {sample}")

    non_numeric = [(d, v) for d, v in values.items() if not isinstance(v, (int, float)) and v is not None]
    if non_numeric:
        print(f" Non-numeric values found for parameter '{param}': {non_numeric}")


Parameter: T2M, Number of towns: 212
 sample: [('20260101', 20.01), ('20260102', 20.35)]
Parameter: T2M_MAX, Number of towns: 212
 sample: [('20260101', 24.54), ('20260102', 26.29)]
Parameter: T2M_MIN, Number of towns: 212
 sample: [('20260101', 15.52), ('20260102', 14.88)]
Parameter: PRECTOTCORR, Number of towns: 212
 sample: [('20260101', 0.43), ('20260102', 0.01)]
Parameter: WS2M, Number of towns: 212
 sample: [('20260101', 1.27), ('20260102', 1.39)]
Parameter: ALLSKY_SFC_SW_DWN, Number of towns: 212
 sample: [('20260101', 25.42), ('20260102', 25.21)]
Parameter: RH2M, Number of towns: 212
 sample: [('20260101', 73.24), ('20260102', 66.17)]
Parameter: GWETTOP, Number of towns: 212
 sample: [('20260101', 0.83), ('20260102', 0.82)]


Checking whether the parameters within one file share the exact same set of dates

In [10]:
date_sets = {param: set(values.keys()) for param, values in data.items()}
first_param = list(date_sets.keys())[0]
for param, dates in date_sets.items():
    if dates != date_sets[first_param]:
        print(f"⚠️ {param} has different dates than {first_param}")
print("All parameters aligned" if all(d == date_sets[first_param] for d in date_sets.values()) else "Mismatch found")

All parameters aligned


Sanity checks value ranges per parameter (catches unit weirdness or corrupted values)
We expect the following from the value checks:
- Temperature (Celsius) - 5-35
- Precipitation - 0-50-100 on a heavy rain day
- T2M: shows a max of 300

In [11]:
for param, values in data.items():
    vals = list(values.values())
    print(f"{param}: min={min(vals):.2f}, max={max(vals):.2f}, mean={sum(vals)/len(vals):.2f}")

T2M: min=17.48, max=21.75, mean=19.91
T2M_MAX: min=19.02, max=28.09, mean=24.90
T2M_MIN: min=11.53, max=17.75, mean=15.35
PRECTOTCORR: min=0.00, max=110.73, mean=8.17
WS2M: min=0.71, max=4.20, mean=1.63
ALLSKY_SFC_SW_DWN: min=12.20, max=28.26, mean=20.44
RH2M: min=55.28, max=90.61, mean=74.93
GWETTOP: min=0.57, max=0.97, mean=0.81


# Transforming

In [12]:
from src.etl.transform import load_county_data

bungoma_merged = load_county_data("bungoma")
print(list(bungoma_merged.keys()))
print(len(bungoma_merged[list(bungoma_merged.keys())[0]]))  # Print the number of entries for the first parameter

['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'WS2M', 'ALLSKY_SFC_SW_DWN', 'RH2M', 'GWETTOP']
1308


In [13]:
from src.etl.transform import reshape_to_dataframe
from src.etl.extract import get_county_files

county_names = ["bungoma", "uasin_gishu", "transzoia", "nakuru"]
weather_df = []
county_data_frames = []

for county in county_names:
    merged_county_data = load_county_data(county)
    county_df = reshape_to_dataframe(merged_county_data, county)
    county_df["county"] = county
    county_data_frames.append(county_df)


Checking summary and shape then storing into processed before storing into sqlite

In [14]:
weather_df = pd.concat(county_data_frames, ignore_index=True)

In [15]:
print("Checking the shape of the dataframe")
weather_df.shape
print("Checking the information")
weather_df.info()


Checking the shape of the dataframe
Checking the information
<class 'pandas.DataFrame'>
RangeIndex: 5232 entries, 0 to 5231
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               5232 non-null   datetime64[us]
 1   T2M                5232 non-null   float64       
 2   T2M_MAX            5232 non-null   float64       
 3   T2M_MIN            5232 non-null   float64       
 4   PRECTOTCORR        5232 non-null   float64       
 5   WS2M               5232 non-null   float64       
 6   ALLSKY_SFC_SW_DWN  5232 non-null   float64       
 7   RH2M               5232 non-null   float64       
 8   GWETTOP            5232 non-null   float64       
 9   county             5232 non-null   str           
dtypes: datetime64[us](1), float64(8), str(1)
memory usage: 408.9 KB


## Normalizing the data 
This allows the data to be more targeted when accessing information related to counties

In [16]:
from src.etl.transform import counties_data, seasons_data, build_daily_weather_df, monthly_summary

county_names = {
    "bungoma": "Bungoma, Kenya",
    "uasin_gishu": "Uasin Gishu, Kenya",
    "transzoia": "Trans nzoia, Kenya",
    "nakuru": "Nakuru, Kenya",
}

counties_df = counties_data(county_names)
seasons_df = seasons_data()
daily_weather_df = build_daily_weather_df(weather_df, counties_df, seasons_df)
monthly_summary_df = monthly_summary(daily_weather_df)

In [17]:
print("Checking counties....")
print(counties_df)
print("Checking seasons....")
print(seasons_df)
print("Checking daily weather data ...")
print(daily_weather_df.head(5))
print("Monthly summary...")
print(monthly_summary_df.head(5))

Checking counties....
   county_id  county_name  latitude  longitude
0          1      bungoma  0.782925  34.719168
1          2  uasin_gishu  0.477194  35.305060
2          3    transzoia  1.045458  34.979044
3          4       nakuru -0.280272  36.071205
Checking seasons....
   season_id   season_name  start_month  end_month
0          1    long_rains            3          5
1          2  dry_season_1            6          9
2          3   short_rains           10         12
3          4  dry_season_2            1          2
Checking daily weather data ...
        date  temp_avg_c  temp_max_c  temp_min_c  precip_mm  wind_speed_ms  \
0 2023-01-01       21.69       29.19       15.17       0.00           3.50   
1 2023-01-02       22.01       29.58       15.21       0.01           3.42   
2 2023-01-03       21.22       28.28       15.83       0.19           2.93   
3 2023-01-04       22.23       30.08       15.07       0.04           3.00   
4 2023-01-05       22.62       30.12       16

In [18]:
monthly_summary_df.head(10)

,county_id,year,month,avg_temp_c,total_precip_mm,avg_humidity_pct,season_id
0,1,2023,1,23.376452,8.70,52.669677,4
1,1,2023,2,25.564286,12.35,46.676071,4
2,1,2023,3,23.102258,529.25,69.527742,1
3,1,2023,4,21.978333,258.21,79.884333,1
4,1,2023,5,21.919355,201.97,80.101613,1
5,1,2023,6,21.160667,276.12,82.728000,2
6,1,2023,7,20.969032,129.81,79.753226,2
7,1,2023,8,21.953548,122.03,74.859032,2
8,1,2023,9,22.120000,223.55,78.635000,2
9,1,2023,10,22.761290,130.10,76.728387,3


In [19]:
print(counties_df.shape)        # (4, 4)
print(seasons_df.shape)         # (4, 4)
print(daily_weather_df.shape)   # (5232, 11)
print(monthly_summary_df.shape) # (~144, 7)

daily_weather_df.dtypes
daily_weather_df.isna().sum()

(4, 4)
(4, 4)
(5232, 11)
(172, 7)


date                      0
temp_avg_c                0
temp_max_c                0
temp_min_c                0
precip_mm                 0
wind_speed_ms             0
solar_radiation_kwh_m2    0
humidity_pct              0
soil_moisture_top         0
county_id                 0
season_id                 0
dtype: int64

# Load data
1. Step one is to create the engine then initialize the schema in Neon DB for all our tables

In [20]:
from src.etl.load import get_engine, init_schema

engine = get_engine()
init_schema(engine)

Database schema initialized successfully.


2. We load the counties and seasons tables with data since its static and would likely not changes as much. On the other hand, county_monthly_summary and daily_weather are derived hence we will insert and update the respective tables.

In [22]:
from src.etl.load import load_to_database, upsert_to_database
# load_to_database(engine, counties_df, table_name="counties")
# load_to_database(engine, seasons_df, table_name="seasons")

# Upsert daily weather and county summary data to the database
# This allows us to update existing records and insert new ones without losing any data.
# upsert_to_database(engine, daily_weather_df, table_name="daily_weather", conflict_columns=["date", "county_id"])
upsert_to_database(engine, monthly_summary_df, table_name="county_monthly_summary", conflict_columns=["year", "month", "county_id"])

Data upserted to table 'county_monthly_summary' successfully.


3. Checking that the data was created in the database

In [24]:
print(pd.read_sql("SELECT COUNT(*) FROM counties", engine))
print(pd.read_sql("SELECT COUNT(*) FROM daily_weather", engine))
print(pd.read_sql("SELECT county_id, COUNT(*) FROM daily_weather GROUP BY county_id", engine))
print(pd.read_sql("SELECT COUNT(*) FROM county_monthly_summary", engine))

   count
0      4
   count
0   5232
  county_id  count
0         3   1308
1         2   1308
2         1   1308
3         4   1308
   count
0    172
